# Lab3 - Multimodal Feature Builder (Videos + Whisper text + Sensorial)

This notebook is prepared to run on the cluster with the same environment and hardware assumptions used in `run.sh` and `run_parallel.sh`.

Pipeline goals:
1. Load Whisper transcripts and sensorial data from EXIST 2026 Videos Dataset.
2. Extract text embeddings with XLM-RoBERTa.
3. Extract keyframes with PySceneDetect.
4. Build early-fusion vectors by concatenating text + video + sensorial features.
5. Save artifacts for the three training configurations (ES, EN, ES+EN).


In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.impute import SimpleImputer

from transformers import AutoTokenizer, AutoModel

from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector

import cv2


In [ ]:
# Cluster configuration copied from run.sh / run_parallel.sh
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_SEQUENTIAL = 'shard:6'  # run.sh
SLURM_GRES_PARALLEL = 'shard:4'    # run_parallel.sh
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

DATA_ROOT = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset')
TRAIN_JSON_PATH = DATA_ROOT / 'training' / 'EXIST2026_training.json'
VIDEO_ROOT = DATA_ROOT / 'training'

PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
KEYFRAME_DIR = PROJECT_ROOT / 'keyframes'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
KEYFRAME_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', 'not set'))
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


In [ ]:
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

records = []
for sample_id, payload in raw_data.items():
    rec = {'sample_id': str(sample_id)}
    rec.update(payload)
    records.append(rec)

df = pd.DataFrame(records)


def majority_task3_1(label_list):
    vals = [x for x in label_list if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]


df['label_task3_1_majority'] = df['labels_task3_1'].apply(majority_task3_1)
df['y'] = df['label_task3_1_majority'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)

df['video_abs_path'] = df['path_video'].apply(lambda p: str((VIDEO_ROOT / p).resolve()))

print('Total samples:', len(df))
print('Languages:', df['lang'].value_counts().to_dict())
print('Label distribution:', df['label_task3_1_majority'].value_counts().to_dict())

df[['sample_id', 'lang', 'split', 'text', 'path_video', 'y']].head(3)


In [ ]:
# Flatten nested sensorial metrics by averaging values across users per modality.
def flatten_sensorial(sensorial_obj):
    if not isinstance(sensorial_obj, dict):
        return {}

    result = {}
    modalities = sensorial_obj.get('modalities', {})
    for mod_name, mod_payload in modalities.items():
        by_user = mod_payload.get('by_user', {}) if isinstance(mod_payload, dict) else {}
        agg = {}
        count = {}

        for _, feat_map in by_user.items():
            if not isinstance(feat_map, dict):
                continue
            for feat_name, feat_val in feat_map.items():
                if feat_val is None:
                    continue
                if isinstance(feat_val, (int, float, np.integer, np.floating)):
                    key = f'{mod_name.lower()}__{feat_name}'
                    agg[key] = agg.get(key, 0.0) + float(feat_val)
                    count[key] = count.get(key, 0) + 1

        for k, v in agg.items():
            result[k] = v / max(count[k], 1)

    return result


sensorial_series = df['sensorial'].apply(flatten_sensorial)
sensorial_df = pd.json_normalize(sensorial_series).astype(float)

if sensorial_df.shape[1] == 0:
    raise RuntimeError('No sensorial features were extracted.')

imputer = SimpleImputer(strategy='constant', fill_value=0.0)
sensorial_matrix = imputer.fit_transform(sensorial_df)
sensorial_columns = sensorial_df.columns.tolist()

print('Sensorial matrix shape:', sensorial_matrix.shape)
print('Sensorial feature count:', len(sensorial_columns))


In [ ]:
# XLM-R text embeddings from Whisper transcripts in field `text`.
TEXT_MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 256
BATCH_SIZE = 16

texts = df['text'].fillna('').astype(str).tolist()

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_model.eval()

all_embeds = []

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='Text embeddings'):
    batch_texts = texts[start:start + BATCH_SIZE]
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        out = text_model(**enc)
        cls_emb = out.last_hidden_state[:, 0, :]

    all_embeds.append(cls_emb.cpu().numpy())

text_matrix = np.vstack(all_embeds).astype(np.float32)
print('Text matrix shape:', text_matrix.shape)


In [ ]:
# Keyframe extraction with PySceneDetect + lightweight frame descriptors.
def extract_keyframes(video_path, output_dir, threshold=30.0, max_keyframes=6):
    output_dir.mkdir(parents=True, exist_ok=True)

    vm = VideoManager([str(video_path)])
    sm = SceneManager()
    sm.add_detector(ContentDetector(threshold=threshold))

    try:
        vm.start()
        sm.detect_scenes(frame_source=vm)
        scenes = sm.get_scene_list()
    finally:
        vm.release()

    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return []

    frame_indices = []
    if len(scenes) == 0:
        frame_indices = [frame_count // 2]
    else:
        for start_tc, end_tc in scenes:
            s = start_tc.get_frames()
            e = end_tc.get_frames()
            frame_indices.append((s + e) // 2)

    frame_indices = frame_indices[:max_keyframes]
    keyframe_paths = []

    for idx_num, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        frame_path = output_dir / f'kf_{idx_num:02d}_{int(frame_idx)}.jpg'
        cv2.imwrite(str(frame_path), frame)
        keyframe_paths.append(frame_path)

    cap.release()
    return keyframe_paths


def keyframe_stats(image_paths):
    if not image_paths:
        return {
            'video__kf_count': 0.0,
            'video__r_mean': 0.0,
            'video__g_mean': 0.0,
            'video__b_mean': 0.0,
            'video__r_std': 0.0,
            'video__g_std': 0.0,
            'video__b_std': 0.0,
            'video__sharpness': 0.0,
        }

    means = []
    stds = []
    sharp = []

    for p in image_paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        means.append(rgb.reshape(-1, 3).mean(axis=0))
        stds.append(rgb.reshape(-1, 3).std(axis=0))
        sharp.append(float(cv2.Laplacian(rgb, cv2.CV_64F).var()))

    if not means:
        return {
            'video__kf_count': 0.0,
            'video__r_mean': 0.0,
            'video__g_mean': 0.0,
            'video__b_mean': 0.0,
            'video__r_std': 0.0,
            'video__g_std': 0.0,
            'video__b_std': 0.0,
            'video__sharpness': 0.0,
        }

    means = np.array(means)
    stds = np.array(stds)

    return {
        'video__kf_count': float(len(means)),
        'video__r_mean': float(means[:, 0].mean()),
        'video__g_mean': float(means[:, 1].mean()),
        'video__b_mean': float(means[:, 2].mean()),
        'video__r_std': float(stds[:, 0].mean()),
        'video__g_std': float(stds[:, 1].mean()),
        'video__b_std': float(stds[:, 2].mean()),
        'video__sharpness': float(np.mean(sharp)),
    }


video_feats = []
for row in tqdm(df.itertuples(index=False), total=len(df), desc='Keyframes + video stats'):
    sample_dir = KEYFRAME_DIR / str(row.sample_id)
    kfs = extract_keyframes(Path(row.video_abs_path), sample_dir)
    stats = keyframe_stats(kfs)
    stats['sample_id'] = str(row.sample_id)
    video_feats.append(stats)

video_df = pd.DataFrame(video_feats).sort_values('sample_id').reset_index(drop=True)
video_feature_cols = [c for c in video_df.columns if c != 'sample_id']
video_matrix = video_df[video_feature_cols].to_numpy(dtype=np.float32)

print('Video matrix shape:', video_matrix.shape)


In [ ]:
meta_df = df[['sample_id', 'lang', 'split', 'y', 'label_task3_1_majority']].copy()
meta_df = meta_df.sort_values('sample_id').reset_index(drop=True)

sensorial_out_df = pd.DataFrame(sensorial_matrix, columns=sensorial_columns)
sensorial_out_df['sample_id'] = df['sample_id'].values
sensorial_out_df = sensorial_out_df.sort_values('sample_id').reset_index(drop=True)

text_out_df = pd.DataFrame(text_matrix)
text_out_df['sample_id'] = df['sample_id'].values
text_out_df = text_out_df.sort_values('sample_id').reset_index(drop=True)

video_df = video_df.sort_values('sample_id').reset_index(drop=True)

assert list(meta_df['sample_id']) == list(sensorial_out_df['sample_id']) == list(text_out_df['sample_id']) == list(video_df['sample_id'])

X_text = text_out_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)
X_video = video_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)
X_sensor = sensorial_out_df.drop(columns=['sample_id']).to_numpy(dtype=np.float32)

X_fusion = np.hstack([X_text, X_video, X_sensor]).astype(np.float32)
y = meta_df['y'].to_numpy(dtype=np.int64)
langs = meta_df['lang'].astype(str).to_numpy()
sample_ids = meta_df['sample_id'].astype(str).to_numpy()

print('Early fusion shape:', X_fusion.shape)
print('Text dim:', X_text.shape[1], '| Video dim:', X_video.shape[1], '| Sensor dim:', X_sensor.shape[1])

np.savez_compressed(
    ARTIFACTS_DIR / 'fusion_features.npz',
    X_fusion=X_fusion,
    y=y,
    langs=langs,
    sample_ids=sample_ids,
    text_dim=np.array([X_text.shape[1]], dtype=np.int64),
    video_dim=np.array([X_video.shape[1]], dtype=np.int64),
    sensor_dim=np.array([X_sensor.shape[1]], dtype=np.int64),
)

meta_df.to_csv(ARTIFACTS_DIR / 'sample_metadata.csv', index=False)
video_df.to_csv(ARTIFACTS_DIR / 'video_features.csv', index=False)
sensorial_out_df.to_csv(ARTIFACTS_DIR / 'sensorial_features.csv', index=False)

print('Saved artifacts in:', ARTIFACTS_DIR)


## Next step

Run one of the training notebooks:
- `02_train_classifier_es.ipynb`
- `03_train_classifier_en.ipynb`
- `04_train_classifier_es_en.ipynb`
